# 18 — Audit aggregation sensitivity and group-relative anomalies

**Optional, after completed 05 and 11.** Use the verified project kernel, then **Restart Kernel and Run All**. No preparation, classifier tuning, source correction or download runs here. Keep the matching local run and deployed detector.

Two bounded questions:
1. What is the exact label agreement between observed-record and equal-premise medians in 05's **training-pair diagnostic**?
2. How do review flags change if the six existing anomaly features are centred/scaled within food group using **training data only**, before fitting one pooled Isolation Forest with the same settings?

This audits different definitions of unusualness. More evenly distributed flags are **not automatically better**, and zero flags need not be a defect. No anomaly ground truth or classifier AP gain is claimed. The existing 10% labels, detector, classifier, deployment pin and dashboard remain unchanged. No test rows or November outcomes are read. Outputs go to a new timestamp under `research/18_label_and_anomaly_audit/` in the active run.

In [1]:
import sys
from pathlib import Path
REPOSITORY_ROOT = Path.cwd().resolve()
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
if not (REPOSITORY_ROOT / "pricecatcher_config.py").is_file():
    raise FileNotFoundError("Open this notebook inside the FYP repository.")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
import ast
import hashlib
import json
from datetime import datetime, timezone
from importlib import metadata, reload
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import pricecatcher_config as cfg
cfg = reload(cfg)

def file_hash(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

pin_path = cfg.PROJECT_ROOT / "11_deployment" / "deployment_selection.json"
link_path = pin_path.with_name("anomaly_selection.json")
if not pin_path.is_file() or not link_path.is_file():
    raise FileNotFoundError("Complete the matching Notebook 11 first. This audit never creates a deployment pin.")
pin, study_dir, study = cfg.dashboard_candidate()  # Existing pin verified, never substituted.
link = json.loads(link_path.read_text())
manifest_path = cfg.resolve_input(link["manifest_path"])
if link["deployment_id"] != pin["deployment_id"] or file_hash(manifest_path) != link["manifest_sha256"]:
    raise ValueError("Detector link and deployment provenance disagree.")
baseline_manifest = json.loads(manifest_path.read_text())
baseline_dir = manifest_path.parent
model_path = baseline_dir / "isolation_forest.joblib"
scores_path = baseline_dir / "anomaly_scores.parquet"
if file_hash(model_path) != baseline_manifest["model_sha256"] or file_hash(scores_path) != link["scores_sha256"]:
    raise ValueError("Existing detector/model scores changed. Restore matching originals.")
for package in ["scikit-learn", "numpy", "pandas", "joblib"]:
    if metadata.version(package) != baseline_manifest["package_versions"][package]:
        raise ValueError(f"Use the saved detector's {package} version for this comparison.")
prepared = cfg.PROJECT_ROOT / "experiments" / "holdout" / "05_modelling"
settings_path = prepared / "experiment_settings.json"
settings = json.loads(settings_path.read_text())
train_path = prepared / "train.parquet"
if (file_hash(train_path) != baseline_manifest["training_input_sha256"]
        or file_hash(train_path) != link["training_input_sha256"]):
    raise ValueError("Anomaly training snapshot differs.")
aggregation_path = prepared / "training_aggregation_coverage_comparison.csv"
paths = [pin_path, link_path, manifest_path, model_path, scores_path, settings_path,
         train_path, prepared / "validation.parquet", aggregation_path, Path(cfg.__file__)]
input_hashes = {cfg.input_reference(p): file_hash(p) for p in paths}
print("Classifier pin (unchanged):", pin["candidate"])
print("Anomaly reference:", baseline_dir)

Classifier pin (unchanged): Longer-history LightGBM (tuned)
Anomaly reference: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/11_anomalies/2fae5011a2a6495c


## 1. Reconstruct the 2×2 training-label agreement

Use 05's saved aggregation diagnostic; this is **not a new price/label reconstruction**, a verified-error count or a performance comparison. The population precedes modelling-history eligibility and can be larger than the final training table. Equal-premise weighting changes observation weights; it does not isolate changing-shop composition. The separate matched-premise diagnostic addresses a different question.

Given original positives A, equal-premise positives B and disagreements D, both-positive pairs are (A+B−D)/2. Show all four cells and their correct denominators. Do not report D/A as the fraction of all labels that changed.

In [2]:
def label_agreement(row):
    counts = np.array([row[k] for k in ["training_pairs", "observed_median_surges", "equal_premise_surges", "labels_disagree"]], dtype=float)
    if not np.isfinite(counts).all() or not np.equal(counts, np.floor(counts)).all():
        raise ValueError("Diagnostic counts must be finite integers.")
    N, A, B, D = map(int, counts)
    if N <= 0 or any(value < 0 or value > N for value in [A, B, D]) or (A+B-D) % 2:
        raise ValueError("Inconsistent aggregation diagnostic counts.")
    both = (A+B-D) // 2
    cells = [(1, 1, both), (1, 0, A-both), (0, 1, B-both), (0, 0, N-A-B+both)]
    if any(count < 0 for _, _, count in cells) or sum(count for _, _, count in cells) != N:
        raise ValueError("Invalid label agreement table.")
    return [{"population": row["population"], "original_label": a, "equal_premise_label": b,
             "pairs": count} for a, b, count in cells]

aggregation = pd.read_csv(aggregation_path)
if aggregation.empty or aggregation.population.duplicated().any():
    raise ValueError("Expected one aggregation diagnostic per named population.")
agreement_rows, sensitivity_rows = [], []
for _, row in aggregation.iterrows():
    counts = label_agreement(row)
    agreement_rows.extend(counts)
    lost = counts[1]["pairs"]
    sensitivity_rows.append({"population": row.population, "training_pairs": int(row.training_pairs),
        "original_surges": int(row.observed_median_surges), "equal_premise_surges": int(row.equal_premise_surges),
        "original_surges_becoming_negative": lost, "original_negatives_becoming_positive": counts[2]["pairs"],
        "all_pairs_disagreement_pct": 100 * row.labels_disagree / row.training_pairs,
        "original_surges_becoming_negative_pct": 100 * lost / row.observed_median_surges if row.observed_median_surges else np.nan})
agreement = pd.DataFrame(agreement_rows)
label_sensitivity = pd.DataFrame(sensitivity_rows)
display(agreement)
display(label_sensitivity)
print("Sensitivity is not proof that either definition is wrong; the supervised target remains unchanged.")

,population,original_label,equal_premise_label,pairs
0,main_coverage,1,1,17257
1,main_coverage,1,0,4407
2,main_coverage,0,1,3838
3,main_coverage,0,0,192073
4,stronger_coverage,1,1,8688
5,stronger_coverage,1,0,4117
6,stronger_coverage,0,1,3262
7,stronger_coverage,0,0,79610


,population,training_pairs,original_surges,equal_premise_surges,original_surges_becoming_negative,original_negatives_becoming_positive,all_pairs_disagreement_pct,original_surges_becoming_negative_pct
0,main_coverage,217575,21664,21095,4407,3838,3.789498,20.342504
1,stronger_coverage,95677,12805,11950,4117,3262,7.712407,32.151503


Sensitivity is not proof that either definition is wrong; the supervised target remains unchanged.


## 2. Reuse the existing six-feature calculation and fixed detector

Load only `anomaly_inputs` and its literal feature declarations from 11, not notebook execution or deployment code. Verify formulas/settings against the deployed detector. Read train/validation input features and dates only; surge labels and future price/coverage columns are not loaded. Validation is scored with frozen training transformations, not used to choose settings.

In [3]:
notebook_11 = REPOSITORY_ROOT / "11_anomaly_detection.ipynb"
definitions, declarations = [], {}
for cell in json.loads(notebook_11.read_text())["cells"]:
    if cell["cell_type"] != "code":
        continue
    for node in ast.parse("".join(cell["source"])).body:
        if isinstance(node, ast.FunctionDef) and node.name == "anomaly_inputs":
            definitions.append(node)
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in {"RAW_FEATURES", "FEATURE_FORMULAS"}:
                    declarations[target.id] = ast.literal_eval(node.value)
if len(definitions) != 1 or set(declarations) != {"RAW_FEATURES", "FEATURE_FORMULAS"}:
    raise ValueError("Notebook 11 feature declarations changed; review compatibility.")
RAW_FEATURES, FEATURE_FORMULAS = declarations["RAW_FEATURES"], declarations["FEATURE_FORMULAS"]
if FEATURE_FORMULAS != baseline_manifest["protocol"]["feature_formulas"]:
    raise ValueError("Existing detector uses different feature formulas.")
helper_tree = ast.Module(body=definitions, type_ignores=[])
helper_sha256 = hashlib.sha256(ast.dump(helper_tree).encode()).hexdigest()
exec(compile(helper_tree, str(notebook_11), "exec"), globals())
PARAMETERS = baseline_manifest["protocol"]["parameters"]
baseline = joblib.load(model_path)
if any(baseline.named_steps["detector"].get_params()[key] != value for key, value in PARAMETERS.items()):
    raise ValueError("Detector parameters and manifest disagree.")
IDENTIFIERS = cfg.GROUP_KEYS + ["week_start", "next_week_end"]

def read_features(period):
    frame = pd.read_parquet(prepared / (period + ".parquet"), columns=IDENTIFIERS + RAW_FEATURES)
    frame["week_start"] = pd.to_datetime(frame.week_start)
    frame["next_week_end"] = pd.to_datetime(frame.next_week_end)
    cutoff = pd.Timestamp(settings["train_end" if period == "train" else "validation_end"])
    if (frame.empty or frame[IDENTIFIERS].isna().any().any()
            or frame.duplicated(cfg.GROUP_KEYS + ["week_start"]).any()
            or not frame.week_start.dt.dayofweek.eq(0).all()
            or not frame.next_week_end.eq(frame.week_start + pd.Timedelta(days=13)).all()
            or not frame.next_week_end.le(cutoff).all()):
        raise ValueError("Invalid or boundary-crossing anomaly inputs.")
    if period == "validation" and not (frame.week_start + pd.Timedelta(days=6)).gt(pd.Timestamp(settings["train_end"])).all():
        raise ValueError("Validation overlaps training.")
    return frame

train = read_features("train")
X_train = anomaly_inputs(train)

## 3. Fit group-relative centring/scaling on training only

For each food group/feature, use its training median and interquartile range (IQR). Require 30 observed feature values before using a group-specific estimate. With inadequate support, use the global training centre/scale. A zero group IQR uses the global training IQR; a zero global IQR uses scale 1. Missing feature values remain missing until the pipeline's training-fitted imputer.

This changes the question from unusual patterns in the pooled population towards unusualness relative to a food group's reference. It is not a correction to source prices and does not guarantee within-item anomalies. Keep one pooled detector and the same contamination setting; do not impose per-group flag quotas.

In [4]:
MIN_VALID_PER_FEATURE = 30

def fit_normalization(X, groups):
    centre = X.median()
    if centre.isna().any():
        raise ValueError("Anomaly training needs observations for each feature.")
    scale = X.quantile(0.75) - X.quantile(0.25)
    global_stats = pd.DataFrame({"centre": centre, "scale": scale.where(scale.gt(0), 1.0)})
    rows = []
    for group in sorted(groups.unique()):
        part = X.loc[groups.eq(group)]
        for feature in X:
            observed = part[feature].dropna()
            enough = len(observed) >= MIN_VALID_PER_FEATURE
            group_iqr = observed.quantile(0.75) - observed.quantile(0.25) if enough else np.nan
            use_group_scale = enough and group_iqr > 0
            rows.append({"project_group": group, "feature": feature, "training_rows": len(part),
                "observed_values": len(observed),
                "centre": float(observed.median() if enough else global_stats.loc[feature, "centre"]),
                "scale": float(group_iqr if use_group_scale else global_stats.loc[feature, "scale"]),
                "group_centre_used": enough, "group_scale_used": bool(use_group_scale)})
    return global_stats, pd.DataFrame(rows)

def normalize(X, groups, global_stats, group_stats):
    result = X.copy()
    for feature in X:
        stats = group_stats.loc[group_stats.feature.eq(feature)].set_index("project_group")
        centre = groups.map(stats.centre).fillna(global_stats.loc[feature, "centre"])
        scale = groups.map(stats.scale).fillna(global_stats.loc[feature, "scale"])
        result[feature] = (X[feature] - centre) / scale
    if np.isinf(result.to_numpy()).any():
        raise ValueError("Nonfinite normalized anomaly input.")
    return result

global_stats, group_stats = fit_normalization(X_train, train.project_group)
normalized_train = normalize(X_train, train.project_group, global_stats, group_stats)
alternative = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                        ("detector", IsolationForest(**PARAMETERS))])
alternative.fit(normalized_train)
display(group_stats)
print("One alternative pooled detector fitted on original 2025 training only.")

,project_group,feature,training_rows,observed_values,centre,scale,group_centre_used,group_scale_used
0,Chicken,current_change_pct,19732,19732,0.000000,0.529101,True,True
1,Chicken,change_2w_pct,19732,19456,0.000000,2.454353,True,True
2,Chicken,change_4w_pct,19732,18564,0.000000,3.822489,True,True
3,Chicken,deviation_4w_pct,19732,19732,0.000000,2.778111,True,True
4,Chicken,within_week_variation_pct,19732,19732,2.785296,5.303383,True,True
5,Chicken,history_variation_4w_pct,19732,19732,1.539601,3.142826,True,True
6,Cooking Oil,current_change_pct,12855,12855,0.000000,1.000000,True,False
7,Cooking Oil,change_2w_pct,12855,10974,0.000000,3.392742,True,False
8,Cooking Oil,change_4w_pct,12855,10254,0.000000,8.107328,True,False
9,Cooking Oil,deviation_4w_pct,12855,12855,0.000000,5.514713,True,False


One alternative pooled detector fitted on original 2025 training only.


## 4. Compare rates, overlap and training review examples

Reproduce the saved baseline's train/validation scores before comparing it. Report rates with denominators and flag shares. Overlap measures agreement, **not accuracy**. The two score scales are not directly comparable. No supervised surge labels are used to tune the detector.

Export the top 20 flagged **training** rows from each method (at most 40 entries, possibly overlapping) for qualitative review. Every case starts unreviewed; do not invent confirmations. Inspect raw price history and independent source evidence where available. Review notes are not ground truth merely because they are written down.

In [5]:
cached = pd.read_parquet(scores_path, columns=cfg.GROUP_KEYS + ["week_start", "split", "anomaly_score", "is_anomaly"])
rate_rows, overlap_rows, score_tables, review_tables = [], [], [], []
for period in ["train", "validation"]:
    frame = train if period == "train" else read_features(period)
    X = X_train if period == "train" else anomaly_inputs(frame)
    Z = normalize(X, frame.project_group, global_stats, group_stats)
    result = frame[IDENTIFIERS].copy()
    result["split"] = period
    for method, detector, inputs in [("pooled", baseline, X), ("group_normalized", alternative, Z)]:
        result[method + "_score"] = -detector.score_samples(inputs)
        result[method + "_flag"] = detector.predict(inputs) == -1
        for group, positions in [("ALL", np.arange(len(frame)))] + list(frame.groupby("project_group").indices.items()):
            flags = result[method + "_flag"].to_numpy()[positions]
            total_flags = int(result[method + "_flag"].sum())
            rate_rows.append({"period": period, "method": method, "project_group": group,
                "rows": len(positions), "flagged": int(flags.sum()), "flag_rate_pct": 100 * flags.mean(),
                "share_of_period_flags_pct": 100 * flags.sum() / total_flags if total_flags else np.nan})
        if period == "train":
            review = frame[IDENTIFIERS + RAW_FEATURES].copy()
            review[list(FEATURE_FORMULAS)] = X
            review["method"] = method
            review["anomaly_score"] = result[method + "_score"]
            review = review.loc[result[method + "_flag"]].nlargest(20, "anomaly_score")
            review["review_status"] = "unreviewed"
            review["evidence_reference"] = ""
            review["review_notes"] = ""
            review_tables.append(review)
    keys = cfg.GROUP_KEYS + ["week_start"]
    expected = cached.loc[cached.split.eq(period)].sort_values(keys).reset_index(drop=True)
    actual = result.sort_values(keys).reset_index(drop=True)
    if expected.duplicated(keys).any() or len(expected) != len(actual):
        raise ValueError("Cached baseline rows differ from the prepared period.")
    pd.testing.assert_frame_equal(actual[keys], expected[keys])
    np.testing.assert_allclose(actual.pooled_score, expected.anomaly_score, rtol=1e-9, atol=1e-11)
    np.testing.assert_array_equal(actual.pooled_flag, expected.is_anomaly)
    both = result.pooled_flag & result.group_normalized_flag
    either = result.pooled_flag | result.group_normalized_flag
    overlap_rows.append({"period": period, "rows": len(result), "both_flagged": int(both.sum()),
        "pooled_only": int((result.pooled_flag & ~result.group_normalized_flag).sum()),
        "group_normalized_only": int((~result.pooled_flag & result.group_normalized_flag).sum()),
        "flag_jaccard": both.sum() / either.sum() if either.any() else np.nan})
    score_tables.append(result)
flag_rates = pd.DataFrame(rate_rows)
flag_overlap = pd.DataFrame(overlap_rows)
review_candidates = pd.concat(review_tables, ignore_index=True)
display(flag_rates)
display(flag_overlap)
print("Flag redistribution alone is not improvement or anomaly accuracy.")

,period,method,project_group,rows,flagged,flag_rate_pct,share_of_period_flags_pct
0,train,pooled,ALL,195664,1957,1.000184,100.000000
1,train,pooled,Chicken,19732,0,0.000000,0.000000
2,train,pooled,Cooking Oil,12855,2,0.015558,0.102197
3,train,pooled,Eggs,40195,17,0.042294,0.868677
4,train,pooled,Flour,7945,0,0.000000,0.000000
5,train,pooled,Rice,4357,0,0.000000,0.000000
6,train,pooled,Sugar,11976,0,0.000000,0.000000
7,train,pooled,Vegetables,98604,1938,1.965438,99.029126
8,train,group_normalized,ALL,195664,1957,1.000184,100.000000
9,train,group_normalized,Chicken,19732,25,0.126698,1.277466


,period,rows,both_flagged,pooled_only,group_normalized_only,flag_jaccard
0,train,195664,70,1887,1887,0.018210
1,validation,43594,10,629,145,0.012755


Flag redistribution alone is not improvement or anomaly accuracy.


## 5. Save research outputs without replacing the dashboard

The research bundle contains normalisation statistics and the alternative detector, which must be used **together** for inference. It is not a drop-in dashboard model. An independent-evidence review and clear anomaly objective are needed before considering integration. New runs preserve earlier notes/results in their own folders.

In [6]:
for reference, digest in input_hashes.items():
    if file_hash(cfg.resolve_input(reference)) != digest:
        raise ValueError("An upstream input or deployment pointer changed during the audit.")
output = cfg.PROJECT_ROOT / "research" / "18_label_and_anomaly_audit" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output.mkdir(parents=True, exist_ok=False)
for name, table in {"label_agreement.csv": agreement, "label_sensitivity.csv": label_sensitivity,
    "anomaly_group_comparison.csv": flag_rates, "anomaly_flag_overlap.csv": flag_overlap,
    "group_normalization.csv": group_stats, "training_review_candidates.csv": review_candidates}.items():
    table.to_csv(output / name, index=False)
pd.concat(score_tables, ignore_index=True).to_parquet(output / "anomaly_scores.parquet", index=False)
joblib.dump({"model": alternative, "global_stats": global_stats, "group_stats": group_stats,
             "feature_columns": list(FEATURE_FORMULAS)}, output / "research_detector.joblib")
this_nb = json.loads((REPOSITORY_ROOT / "research" / "18_label_and_anomaly_audit.ipynb").read_text())
code_sha256 = hashlib.sha256("\n".join("".join(c["source"]) for c in this_nb["cells"] if c["cell_type"] == "code").encode()).hexdigest()
record = {"status": "completed", "python_version": sys.version.split()[0], "classifier_candidate": pin["candidate"],
    "deployment_id": pin["deployment_id"], "parameters": PARAMETERS, "feature_formulas": FEATURE_FORMULAS,
    "normalization": "training group median/IQR; sparse/zero-scale fallback to global training statistics",
    "minimum_observed_per_group_feature": MIN_VALID_PER_FEATURE, "separate_group_flag_quotas": False,
    "dashboard_changed": False, "surge_target_changed": False, "test_evaluated": False,
    "anomaly_accuracy_established": False, "input_sha256": input_hashes, "feature_helper_sha256": helper_sha256,
    "research_code_sha256": code_sha256, "package_versions": baseline_manifest["package_versions"],
    "output_sha256": {p.name: file_hash(p) for p in output.iterdir()}}
(output / "research_manifest.json").write_text(json.dumps(record, indent=2))
print("Saved:", output)
print("Share label_sensitivity.csv, anomaly_group_comparison.csv and anomaly_flag_overlap.csv.")
print("Review the training shortlist separately; no errors have been confirmed automatically.")

Saved: /Users/alfredyap/Documents/GitHub/FYP/data/runs/run_20261019_013241_618240/research/18_label_and_anomaly_audit/20261103_175044_741020
Share label_sensitivity.csv, anomaly_group_comparison.csv and anomaly_flag_overlap.csv.
Review the training shortlist separately; no errors have been confirmed automatically.


## Interpretation for the report

Report the label table's population and both disagreement directions. A changed label under another weighting policy is not automatically a corrected error. For anomalies, distinguish overall pooled unusualness from group-relative unusualness and inspect rates plus representative examples. Centre/scale choices can redistribute flags without improving validity. Manual inspection of a prioritised shortlist is not an unbiased estimate of accuracy.

The original four-model comparison remains an equal-history algorithm comparison; added history/searches are separate experiments. The detector uses original 2025 inputs, while the deployed classifier may use 2024–2025 history. Current validation results and earlier exposed test outcomes do not become a new independent test. No new November registration or checkpoint substitution is made here.